In [72]:
import pandas as pd
import numpy as np

In [73]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [74]:
df = pd.read_csv("covid_toy.csv")

In [75]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [76]:
df["city"].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [77]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [78]:
# So for age we do nothing it is numerical column
# For gender which is categorical and it is nominal we will use one hot encoding
# for Fever we will use simple imputer to fill the missing values and then we will use ordinal encoding beacuse it is ordinal
# For cough which is categorical and it is ordinal we will use ordinal encoding
# For city which is categorical and it is nominal we will use one hot encoding
# has covid is the target variable so we will do nothing with it

First thing we always need to do Train Test Split

In [79]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=["has_covid"]),df["has_covid"],test_size=0.2,random_state=42)

# In train_test_split first parameter is the features and second parameter is the target variable. 
# We are dropping the target variable from the features. test_size is the size of the test set and random_state is used for reproducibility.


In [80]:
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


Simpute Impute on Fever Column

In [81]:
si = SimpleImputer(strategy="most_frequent")

X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.transform(X_test[['fever']])

In [82]:
X_train_fever.shape


(80, 1)

Ordinal Encoding in Cough column

In [83]:
oe = OrdinalEncoder(categories=[["Mild","Strong"]])

X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.transform(X_test[['cough']])

X_train_cough

array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],

Ordinal Encoding in Gender and City 

In [88]:
ohe = OneHotEncoder(drop='first',sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[["gender","city"]])

X_test_gender_city = ohe.transform(X_test[["gender","city"]])

X_train_gender_city.shape


(80, 4)

Extracting Age Column

In [89]:
X_train_age = X_train.drop(columns=['gender','city','cough','fever']).values

X_test_age = X_test.drop(columns=['gender','city','cough','fever']).values



Cancatinating All the Columns

In [92]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

Instead of all this we use Transformers

In [94]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
63,10,Male,100.0,Mild,Bangalore,No
20,12,Male,98.0,Strong,Bangalore,No
97,20,Female,101.0,Mild,Bangalore,No
54,60,Female,99.0,Mild,Mumbai,Yes
50,19,Male,101.0,Mild,Delhi,Yes


In [99]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OneHotEncoder(drop='first',sparse_output=False),['city','gender']),
    ('tnf3',OrdinalEncoder(categories=[['Mild','Strong']]),['cough'])
],remainder='passthrough')

transformer.fit_transform(X_train)

transformer.transform(X_test)

array([[104.,   0.,   1.,   0.,   0.,   0.,  17.],
       [ 98.,   1.,   0.,   0.,   1.,   0.,  83.],
       [101.,   1.,   0.,   0.,   0.,   1.,  68.],
       [ 99.,   0.,   0.,   0.,   1.,   0.,  72.],
       [102.,   1.,   0.,   0.,   1.,   1.,  20.],
       [103.,   0.,   1.,   0.,   0.,   0.,  50.],
       [ 98.,   0.,   1.,   0.,   0.,   1.,  71.],
       [ 99.,   0.,   0.,   1.,   0.,   0.,  14.],
       [101.,   1.,   0.,   0.,   0.,   0.,  75.],
       [103.,   0.,   1.,   0.,   1.,   0.,  60.],
       [ 98.,   0.,   0.,   0.,   0.,   0.,  64.],
       [101.,   1.,   0.,   0.,   1.,   0.,  15.],
       [ 98.,   0.,   1.,   0.,   1.,   1.,  34.],
       [ 98.,   0.,   1.,   0.,   0.,   0.,  26.],
       [ 99.,   1.,   0.,   0.,   0.,   1.,  59.],
       [101.,   0.,   0.,   1.,   0.,   0.,  65.],
       [100.,   0.,   0.,   0.,   1.,   0.,  80.],
       [101.,   0.,   1.,   0.,   0.,   0.,   8.],
       [ 99.,   0.,   1.,   0.,   0.,   1.,  25.],
       [103.,   0.,   1.,   0.,